In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

In [2]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.1
)

response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is **Paris**.


In [10]:
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from urllib.parse import quote_plus
import os

load_dotenv()

# Database credentials
db_user = os.getenv("DB_USER")
db_password = quote_plus(os.getenv("DB_PASSWORD"))  # URL-encode password
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# Create database connection
db_uri = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
db = SQLDatabase.from_uri(db_uri)

print(f"Connected to database: {db_name}")
print(f"Tables: {db.get_usable_table_names()}")
print(db.table_info)

Connected to database: electronics_store
Tables: ['discounts', 'products']

CREATE TABLE discounts (
	discount_id SERIAL NOT NULL, 
	product_id INTEGER NOT NULL, 
	pct_discount NUMERIC(5, 2), 
	CONSTRAINT discounts_pkey PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_product_id_fkey FOREIGN KEY(product_id) REFERENCES products (product_id), 
	CONSTRAINT discounts_pct_discount_check CHECK (pct_discount >= 0::numeric AND pct_discount <= 100::numeric)
)

/*
3 rows from discounts table:
discount_id	product_id	pct_discount
1	1	10.00
2	4	15.00
3	5	12.00
*/


CREATE TABLE products (
	product_id SERIAL NOT NULL, 
	brand VARCHAR(50) NOT NULL, 
	category VARCHAR(50) NOT NULL, 
	model_name VARCHAR(100) NOT NULL, 
	specs VARCHAR(200), 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	CONSTRAINT products_pkey PRIMARY KEY (product_id), 
	CONSTRAINT products_brand_category_model_name_key UNIQUE NULLS DISTINCT (brand, category, model_name), 
	CONSTRAINT products_price_check CHECK (price >= 100 AN

In [11]:
# Create SQL query prompt template
template = """Based on the table schema below, write a PostgreSQL query to answer the user's question.

Schema:
{schema}

Question: {question}

SQL Query (only return the SQL, no explanation):"""

prompt = ChatPromptTemplate.from_template(template)

# Get schema info
schema = db.get_table_info()

# Create chain
chain = prompt | llm | StrOutputParser()

# Test
question = "How many Apple products are in stock?"
sql_query = chain.invoke({"schema": schema, "question": question})
print(f"Question: {question}\n")
print(f"Generated SQL:\n{sql_query}\n")

# Execute query
result = db.run(sql_query.strip().replace('```sql', '').replace('```', ''))
print(f"Result: {result}")

Question: How many Apple products are in stock?

Generated SQL:
```sql
SELECT SUM(stock_quantity)
FROM products
WHERE brand = 'Apple';
```

Result: [(415,)]


In [15]:
question = "How much is the price of the inventory for all apple products?"
sql_query = chain.invoke({"schema": schema, "question": question})
print(f"Question: {question}\n")
print(f"Generated SQL:\n{sql_query}\n")

# Execute query
result = db.run(sql_query.strip().replace('```sql', '').replace('```', ''))
print(f"Result: {result}")

Question: How much is the price of the inventory for all apple products?

Generated SQL:
```sql
SELECT SUM(price * stock_quantity)
FROM products
WHERE brand = 'Apple';
```

Result: [(275585,)]


In [16]:
question = "If we sell all Apple products today with discounts applied, how much revenue will our store generate?"
sql_query = chain.invoke({"schema": schema, "question": question})
print(f"Question: {question}\n")
print(f"Generated SQL:\n{sql_query}\n")

# Execute query
result = db.run(sql_query.strip().replace('```sql', '').replace('```', ''))
print(f"Result: {result}")

Question: If we sell all Apple products today with discounts applied, how much revenue will our store generate?

Generated SQL:
```sql
SELECT SUM(p.price * (1 - COALESCE(d.pct_discount, 0) / 100) * p.stock_quantity)
FROM products AS p
LEFT JOIN discounts AS d
  ON p.product_id = d.product_id
WHERE
  p.brand = 'Apple';
```

Result: [(Decimal('267397.50000000000000000000'),)]
